In [ ]:
"""
LPG accessibility and usage mapping (Nigeria, 1 km, EPSG:3857)

This script calculates the percentage of the population with access to LPG within 30 minutes motorized travel time,
using a friction surface (minutes per meter) and a set of LPG reseller points. It then applies national usage shares
to estimate LPG usage across the country, redistributing usage near LPG points.

Important note on population raster interpretation:
- Higher pixel values = more people in that pixel.
- Lower pixel values = fewer people.
- Population is used directly as counts (never inverted).

eta for the run: 15 minutes
"""

import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
from rasterio.features import geometry_mask
import sys

DATA_DIR = "dataset_big"

# File paths - ora con DATA_DIR
lpg_points_file = f"{DATA_DIR}/full_lpg_chain_nig_3857.gpkg"
boundary_file = f"{DATA_DIR}/Country_boundaries.geojson"
pop_raster_file = f"{DATA_DIR}/Population.tif"
urban_raster_file = f"{DATA_DIR}/Urban.tif"
friction_file = f"{DATA_DIR}/friction_moto.tif"
output_raster = f"{DATA_DIR}/lpg_use.tif"

# Target CRS and resolution
target_crs = "EPSG:3857"
target_res = 1000  # meters

# Accessibility threshold (minutes)
threshold_min = 30

# National LPG usage shares
urban_share = 0.25468
rural_share = 0.027905

# Friction is in minutes per meter. To get minutes per 1 km pixel, multiply by 1000.
friction_scale = 1000.0

# Urban threshold: values >= 20 considered urban
urban_threshold = 20

# Layer containing LPG resellers in the geopackage
lpg_layer = "resell_and_filling"

# 1. Load Nigeria boundary and create target grid
boundary = gpd.read_file(boundary_file)
boundary = boundary.to_crs(target_crs)
nigeria_geom = boundary.geometry.union_all()

# Bounding box of Nigeria in target CRS
bounds = nigeria_geom.bounds
minx, miny, maxx, maxy = bounds

# Grid dimensions
width = int(np.ceil((maxx - minx) / target_res))
height = int(np.ceil((maxy - miny) / target_res))
transform = rasterio.transform.from_origin(minx, maxy, target_res, target_res)

print("=== Grid setup ===")
print(f"Resolution: {target_res} m, size: {width} x {height} pixels")
print(f"Coverage: {bounds}")

# 2. Load LPG reseller points
resellers = gpd.read_file(lpg_points_file, layer=lpg_layer)
if resellers.crs != target_crs:
    resellers = resellers.to_crs(target_crs)
print(f"Loaded {len(resellers)} LPG reseller points")

# 3. Reproject population raster
with rasterio.open(pop_raster_file) as src:
    pop_reproj = np.zeros((height, width), dtype=np.float32)
    reproject(
        source=rasterio.band(src, 1),
        destination=pop_reproj,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=src.nodata if src.nodata is not None else -99999,
        dst_transform=transform,
        dst_crs=target_crs,
        dst_nodata=np.nan,
        resampling=Resampling.sum
    )
    pop_profile = src.profile
    pop_profile.update({
        'crs': target_crs,
        'transform': transform,
        'width': width,
        'height': height,
        'dtype': 'float32',
        'nodata': np.nan
    })

# 4. Reproject urban raster and binarize
with rasterio.open(urban_raster_file) as src:
    urban_reproj = np.zeros((height, width), dtype=np.uint8)
    reproject(
        source=rasterio.band(src, 1),
        destination=urban_reproj,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=transform,
        dst_crs=target_crs,
        resampling=Resampling.nearest
    )
urban_binary = (urban_reproj >= urban_threshold).astype(np.uint8)

# 5. Reproject friction raster, scale to minutes per pixel (1 km)
with rasterio.open(friction_file) as src:
    friction_reproj = np.zeros((height, width), dtype=np.float32)
    reproject(
        source=rasterio.band(src, 1),
        destination=friction_reproj,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=-9999,
        dst_transform=transform,
        dst_crs=target_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear
    )
friction_reproj = friction_reproj * friction_scale
friction_reproj[friction_reproj <= 0] = np.nan
print("\n=== Friction surface ===")
print(f"Travel time per km: min {np.nanmin(friction_reproj):.2f} min, max {np.nanmax(friction_reproj):.2f} min")

# 6. Mask all rasters with Nigeria boundary
mask_nigeria = geometry_mask(
    [nigeria_geom],
    transform=transform,
    invert=True,
    out_shape=(height, width)
)

pop_clipped = pop_reproj.copy()
pop_clipped[~mask_nigeria] = np.nan

# Population interpretation fix:
# keep population as absolute counts; higher values mean more people.
# clamp rare negative artifacts to 0 (do NOT invert values).
pop_clipped = np.where(np.isnan(pop_clipped), np.nan, np.maximum(pop_clipped, 0.0)).astype(np.float32)

urban_clipped = urban_binary.copy()
urban_clipped[~mask_nigeria] = 0

friction_clipped = friction_reproj.copy()
friction_clipped[~mask_nigeria] = np.nan

total_pop = np.nansum(pop_clipped)
print(f"\n=== Nigeria population (masked) ===")
print(f"Total: {total_pop:,.0f}")
print("Population raster convention: higher value = more people per pixel")

if total_pop == 0:
    sys.exit("ERROR: Population sum is zero after masking. Cannot continue.")

# 7. Build graph for travel time calculation
valid = ~np.isnan(friction_clipped)
rows, cols = np.where(valid)
total_cells = height * width
valid_cells = len(rows)
print(f"\n=== Travel network ===")
print(f"Valid cells (travel possible): {valid_cells} ({valid_cells/total_cells*100:.1f}% of total grid)")
print(f"  (cells with invalid friction are excluded, e.g., water bodies)")

node_id_map = -np.ones((height, width), dtype=int)
node_id_map[rows, cols] = np.arange(valid_cells)

edge_i, edge_j, edge_cost = [], [], []
neighbors = [(-1,0), (1,0), (0,-1), (0,1)]

for r, c in zip(rows, cols):
    node = node_id_map[r, c]
    for dr, dc in neighbors:
        nr, nc = r + dr, c + dc
        if 0 <= nr < height and 0 <= nc < width:
            nnode = node_id_map[nr, nc]
            if nnode != -1:
                cost = (friction_clipped[r, c] + friction_clipped[nr, nc]) / 2.0
                if cost <= 0:
                    continue
                edge_i.append(node); edge_j.append(nnode); edge_cost.append(cost)
                edge_i.append(nnode); edge_j.append(node); edge_cost.append(cost)

num_edges = len(edge_i) // 2
print(f"Connections between cells (undirected edges): {num_edges:,}")
graph = csr_matrix((edge_cost, (edge_i, edge_j)), shape=(valid_cells, valid_cells))

# 8. Compute accessibility from resellers
reseller_cols = ((resellers.geometry.x - transform[2]) / transform[0]).astype(int)
reseller_rows = ((resellers.geometry.y - transform[5]) / transform[4]).astype(int)

source_nodes = []
for r, c in zip(reseller_rows, reseller_cols):
    if 0 <= r < height and 0 <= c < width and node_id_map[r, c] != -1:
        source_nodes.append(node_id_map[r, c])
source_nodes = list(set(source_nodes))
print(f"Source cells containing at least one reseller: {len(source_nodes)} ({len(source_nodes)/valid_cells*100:.1f}% of valid cells (possible to travel) of Nigeria)")

if not source_nodes:
    raise RuntimeError("No reseller falls inside valid cells.")

dist_matrix, _ = dijkstra(graph, directed=False, indices=source_nodes,
                          return_predecessors=True, unweighted=False)
min_distances = np.min(dist_matrix, axis=0)

dist_full = np.full((height, width), np.inf, dtype=np.float32)
dist_full[rows, cols] = min_distances

accessible = (dist_full <= threshold_min) & valid
accessible_cells = np.sum(accessible)
print(f"\n=== Accessibility (within {threshold_min} min travel) ===")
print(f"Accessible cells: {accessible_cells} ({accessible_cells/valid_cells*100:.1f}% of travelable cells)")

# 9. Separate urban and rural accessible areas
urban_accessible = accessible & (urban_clipped == 1)
rural_accessible = accessible & (urban_clipped == 0) & valid
print(f"  Urban accessible cells: {np.sum(urban_accessible)}")
print(f"  Rural accessible cells: {np.sum(rural_accessible)}")

# 10. Compute total and accessible populations
urban_pop_total = np.nansum(pop_clipped[urban_clipped == 1])
rural_pop_total = np.nansum(pop_clipped[urban_clipped == 0])
urban_pop_acc = np.nansum(pop_clipped[urban_accessible])
rural_pop_acc = np.nansum(pop_clipped[rural_accessible])

print("\n=== Population totals ===")
print(f"Urban: {urban_pop_total:,.0f} ({urban_pop_total/total_pop*100:.1f}%)")
print(f"Rural: {rural_pop_total:,.0f} ({rural_pop_total/total_pop*100:.1f}%)")

print("\n=== Population within 30 minutes of a reseller ===")
print(f"Urban accessible: {urban_pop_acc:,.0f} ({urban_pop_acc/urban_pop_total*100:.1f}% of urban pop)")
print(f"Rural accessible: {rural_pop_acc:,.0f} ({rural_pop_acc/rural_pop_total*100:.1f}% of rural pop)")

# 11. Calculate uniform LPG usage percentages
if urban_pop_acc > 0:
    p_urban = (urban_share * urban_pop_total) / urban_pop_acc
else:
    p_urban = 0.0
if rural_pop_acc > 0:
    p_rural = (rural_share * rural_pop_total) / rural_pop_acc
else:
    p_rural = 0.0

print("\n=== LPG usage shares applied to accessible cells ===")
print(f"Urban accessible cells: {p_urban:.4f} (fraction of people using LPG)")
print(f"Rural accessible cells: {p_rural:.4f} (fraction of people using LPG)")

# 12. Create final LPG usage raster
lpg_use = np.zeros((height, width), dtype=np.float32)
lpg_use[urban_accessible] = p_urban
lpg_use[rural_accessible] = p_rural
lpg_use[~mask_nigeria] = -1  # nodata

# 13. Save raster
profile = pop_profile.copy()
profile.update({
    'dtype': 'float32',
    'nodata': -1,
    'compress': 'lzw'
})
with rasterio.open(output_raster, 'w', **profile) as dst:
    dst.write(lpg_use, 1)
print(f"\nSaved raster to: {output_raster}")

# 14. Verification: check that national totals are preserved
pop_valid = pop_clipped[mask_nigeria]
use_valid = lpg_use[mask_nigeria]
avg_actual = np.nansum(pop_valid * use_valid) / np.nansum(pop_valid)
expected = (urban_share * urban_pop_total + rural_share * rural_pop_total) / total_pop

print("\n=== Verification (national weighted average) ===")
print(f"Actual LPG usage   : {avg_actual:.6f}")
print(f"Expected LPG usage : {expected:.6f}")
if np.isclose(avg_actual, expected, rtol=1e-9):
    print("--> Totals preserved: YES")
else:
    print("--> Totals preserved: NO (check calculations)")

=== Grid setup ===
Resolution: 1000 m, size: 1337 x 1087 pixels
Coverage: (297048.38458976545, 475821.52340381127, 1633771.093451338, 1561830.0109293496)
Loaded 2792 LPG reseller points

=== Friction surface ===
Travel time per km: min 0.50 min, max 160.78 min

=== Nigeria population (masked) ===
Total: 205,325,168
Population raster convention: higher value = more people per pixel

=== Travel network ===
Valid cells (travel possible): 940808 (64.7% of total grid)
  (cells with invalid friction are excluded, e.g., water bodies)
Connections between cells (undirected edges): 3,752,838
Source cells containing at least one reseller: 2189 (0.2% of valid cells (possible to travel) of Nigeria)

=== Accessibility (within 30 min travel) ===
Accessible cells: 65909 (7.0% of travelable cells)
  Urban accessible cells: 16300
  Rural accessible cells: 49609

=== Population totals ===
Urban: 90,887,840 (44.3%)
Rural: 114,437,320 (55.7%)

=== Population within 30 minutes of a reseller ===
Urban access